# 06 – MER2024: trích feature một lần (GPU) và tái lập baseline
**Mục đích:** lưu sẵn feature của 5.030 clip Train&Val MER2024 để các phép sàng lọc sau chỉ cần notebook CPU (notebook 07). Notebook cũng tái lập baseline chính thức (5-fold CV, WAF), đo MDE thật và các lát cắt lỗi.

**Cài đặt notebook:** Accelerator = **GPU T4 x2**; **Internet = On**.
1. *Add-ons → Secrets*: bật `HF_TOKEN`, giống notebook 04.
2. Nếu ô "Giải nén" báo `encrypted: True`: thêm secret `MER2024_ZIP_PW`, lấy mật khẩu trong `README_AFTER_APPROVAL.md` của repo MER2024 trên Hugging Face. **Không dán mật khẩu vào chat hay vào code.**
3. *Save Version → Save & Run All*. Thời gian chạy chưa đo được (ước khoảng 1,5–3 giờ trên T4). Notebook tự dừng trước 10 giờ; nếu chưa xong thì chạy lại, nó sẽ resume.

**Output:**
- `/kaggle/working/mer24_cache/`: feature từng clip kèm file nhãn. **Chỉ để trên Kaggle**, dùng làm input cho notebook 07. Không tải về, không chia sẻ (điều khoản MER).
- `/kaggle/working/e06_report.md`: chỉ có số liệu tổng hợp. Tải file này và gửi cho Claude.

In [ ]:
HOURS = 9.0
KOUT, WORK = '/kaggle/working', '/tmp'
CACHE = f'{KOUT}/mer24_cache'

In [ ]:
import os, subprocess, glob, shutil, zipfile
sh = lambda cmd, check=True: subprocess.run(cmd, shell=True, check=check)
sh(f'rm -rf {WORK}/icme && git clone -q --depth 1 -b main https://github.com/ptrnghieu/icme.git {WORK}/icme')
print('code commit:', subprocess.run(f'git -C {WORK}/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout)
sh('pip -q install -U huggingface_hub easyocr', check=False)
from kaggle_secrets import UserSecretsClient
try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    raise SystemExit('HF_TOKEN secret is not attached to THIS notebook: Add-ons -> Secrets -> tick HF_TOKEN (or Add a new secret labelled HF_TOKEN), then Save & Run All again.') from None
os.environ['HF_HOME'] = f'{WORK}/hf'
import torch; print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')), '| cuda:', torch.cuda.is_available())

In [ ]:
from huggingface_hub import hf_hub_download
D = f'{WORK}/mer'; os.makedirs(CACHE, exist_ok=True)
for fn in ['label-disdim.csv', 'label-transcription.csv', 'video-labeled.zip']:
    p = hf_hub_download(repo_id='MERChallenge/MER2024', repo_type='dataset', filename=fn, local_dir=f'{D}/MER2024', token=os.environ['HF_TOKEN'])
    print(fn, f'{os.path.getsize(p) / 1e9:.2f} GB')
for fn in ['label-disdim.csv', 'label-transcription.csv']:
    shutil.copy(f'{D}/MER2024/{fn}', f'{CACHE}/{fn}')   # labels stay on Kaggle with the cache (input of notebook 07)

In [ ]:
# Giải nén (mật khẩu, nếu có, chỉ đọc từ Kaggle secret, không in ra)
Z = f'{D}/MER2024/video-labeled.zip'; V = f'{WORK}/mer24_videos'; os.makedirs(V, exist_ok=True)
enc = any(zi.flag_bits & 0x1 for zi in zipfile.ZipFile(Z).infolist()[:50]); print('encrypted:', enc)
if enc:
    pw = UserSecretsClient().get_secret('MER2024_ZIP_PW')
    cmd = ['7z', 'x', '-y', f'-p{pw}', f'-o{V}', Z] if shutil.which('7z') else ['unzip', '-q', '-o', '-P', pw, Z, '-d', V]
    print('extract return code:', subprocess.run(cmd, capture_output=True, text=True).returncode); del pw
else:
    zipfile.ZipFile(Z).extractall(V)
import pandas as pd
vids = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{V}/**/*.*', recursive=True) if p.lower().endswith(('.mp4', '.avi', '.mkv'))}
names = set(pd.read_csv(f'{CACHE}/label-disdim.csv')['name'].astype(str))
print('video files:', len(vids), '| labelled names with a video:', len(names & vids), '/', len(names))

In [ ]:
subprocess.run(['python', f'{WORK}/icme/explore/mer24/mer24_features.py', '--videos', V, '--labels', f'{CACHE}/label-disdim.csv',
                '--cache', CACHE, '--hours', str(HOURS)], check=False)
print('parts:', len(glob.glob(f'{CACHE}/part_*.npz')))

In [ ]:
n_parts = len(glob.glob(f'{CACHE}/part_*.npz'))
if n_parts * 250 >= len(names):
    S = f'{WORK}/icme/explore/mer24/mer24_screen.py'
    subprocess.run(['python', S, 'text', '--cache', CACHE, '--labels', f'{CACHE}/label-disdim.csv', '--trans', f'{CACHE}/label-transcription.csv'], check=False)
    subprocess.run(['python', S, 'baseline', '--cache', CACHE, '--labels', f'{CACHE}/label-disdim.csv', '--out', f'{KOUT}/e06_report.md'], check=False)
    print(open(f'{KOUT}/e06_report.md').read())
else:
    print(f'extraction not finished ({n_parts} parts); re-run the notebook to resume')